# Clean-Core Summary Analysis

This notebook is designed for the packaged GitHub repo. It reads committed summary CSVs under `results/summary/` and does not require raw JSONL outputs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'results' / 'summary').exists() and (ROOT.parent / 'results' / 'summary').exists():
    ROOT = ROOT.parent
SUMMARY = ROOT / 'results' / 'summary'
print('Root:', ROOT)
print('Summary files:', len(list(SUMMARY.glob('*.csv'))))

## High-Level Signals

Positive values generally mean the interface manipulation changed behavior in the expected direction. Read the `read_as` column before interpreting any row.

In [ ]:
signals = pd.read_csv(SUMMARY / 'clean_core_signal_summary.csv')
signals

In [ ]:
pivot = signals.pivot(index='signal', columns='model', values='value')
ax = pivot.plot(kind='barh', figsize=(10, 5), grid=True)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Clean-core signal summary by model')
ax.set_xlabel('Signal value')
plt.tight_layout()

## Clean 2: Moving Required Minimum

This checks whether median `c_output` rises as the required floor moves from `.31` to `.46` to `.61`.

In [ ]:
floor = pd.read_csv(SUMMARY / 'clean_core_floor_gradient_v2_summary.csv')
floor_small = floor[['model_key', 'context_id', 'baseline_c', 'word', 'records', 'tool_call_rate', 'threshold_valid_rate', 'strict_only_c_rate', 'median_c_output']]
floor_small.head(20)

In [ ]:
tmp = floor.copy()
tmp['floor'] = tmp['context_id'].str.extract(r'floor_(\d+)').astype(float) / 100
plot_df = tmp.groupby(['model_key', 'floor'], as_index=False)['median_c_output'].median()
fig, ax = plt.subplots(figsize=(7, 4))
for model, g in plot_df.groupby('model_key'):
    ax.plot(g['floor'], g['median_c_output'], marker='o', label=model)
ax.plot([0.31, 0.46, 0.61], [0.31, 0.46, 0.61], linestyle='--', color='gray', label='floor')
ax.set_title('Median c_output rises with required floor')
ax.set_xlabel('Required c floor')
ax.set_ylabel('Median c_output')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

## Clean 5: Rule Location

This separates tool-description effects from user-prompt effects. Some cells abstain, so interpret tool-call rates alongside numeric values.

In [ ]:
loc = pd.read_csv(SUMMARY / 'clean_core_rule_location_v2_summary.csv')
loc[['model_key', 'rule_location', 'records', 'abstentions', 'tool_call_rate', 'threshold_valid_rate', 'median_c_output']]

## Progress Snapshot

These are the local run-completion summaries included for context. They are not required to rerun the clean-core configs.

In [ ]:
progress = pd.read_csv(SUMMARY / 'clean_core_progress_by_batch.csv')
progress[['batch_label', 'records', 'progress_pct', 'tool_calls', 'abstentions', 'schema_invalid']]